# Setup

To run this notebook on Colab, a few setup steps are required. Follow along step by step:

1. **Clone the `dlfb` library**  
   First, clone the repository that contains the `dlfb` library.

In [ ]:
%cd /content
!rm -rf ./dlfb-clone/
!git clone "https://github.com/deep-learning-for-biology/deep-learning-for-biology-pytorch.git" dlfb-clone --branch main
%cd dlfb-clone

2. **Install dependencies**  
   Once the library is cloned, install the required dependencies.

In [ ]:
%%bash
pip install -e '.[localization]'

3. **Providion the datasets**  
   You’ll then need to access and download the necessary datasets for this chapter.

In [ ]:
from google.colab import auth

auth.authenticate_user()
# NOTE: exclude models with '--no-models' flag
!dlfb-provision --chapter localization

4. **Load the `dlfb` package**  
   Finally, load the `dlfb` package.  
   - ⚠️ Note: Loading can sometimes be finicky. If you encounter issues, simply **restart the runtime**. All previously downloaded data and installed packages will persist, so you can re-run the load step without repeating everything.

In [ ]:
try:
  import dlfb_pytorch
except ImportError as exc:
  import site
  site.main()
  import dlfb_pytorch

from dlfb_pytorch.utils.display import display

# 6. Learning Spatial Organization Patterns Within Cells


## 6.1. Biology Primer
### 6.1.1. Spatial Organization within the Cell
### 6.1.2. Protein Localization
### 6.1.3. Understanding Protein Localization


## 6.2. Machine Learning Primer
### 6.2.1. Autoencoders
### 6.2.2. Variational Autoencoders
#### 6.2.2.1. Why add randomness?
#### 6.2.2.2. Continuous Latent Space
### 6.2.3. Vector-Quantized Variational Autoencoders (VQ-VAEs)
#### 6.2.3.1. Where Does the Codebook Come From?
#### 6.2.3.2. How Large Should the Codebook Be?
### 6.2.4. Dissecting a VQ-VAE Diagram
### 6.2.5. Training a VQ-VAE


## 6.3. Constructing the Dataset
### 6.3.1. Data Requirements
### 6.3.2. Sourcing the Data
### 6.3.3. Getting a Glimpse of the Dataset


In [ ]:
import numpy as np
import torch

from dlfb_pytorch.localization.dataset.utils import get_dataset
from dlfb_pytorch.utils.context import assets

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = get_dataset(data_path=assets("localization/datasets"))
n_frames = 16
dataset.plot_random_frames(n=n_frames);

In [ ]:
selected_protein = "ACTB"
dataset.plot_random_frames(
  n=n_frames, with_labels=False, rng=rng_frames, gene_symbols=[selected_protein]
);

In [ ]:
dataset.plot_random_frames(n=1, with_labels=False, rng=rng_frames);

### 6.3.4. Implementing a DatasetBuilder Class


In [ ]:
from dlfb_pytorch.localization.dataset.builder import DatasetBuilder

display([DatasetBuilder])

#### 6.3.4.1. Building a First Dataset Instance


In [ ]:
from dlfb_pytorch.localization.dataset import Dataset
from dlfb_pytorch.utils.context import assets

builder = DatasetBuilder(data_path=assets("localization/datasets"))

np.random.seed(42)
dataset_splits: dict[str, Dataset] = builder.build(
  splits={"train": 0.80, "valid": 0.10, "test": 0.10},
  exclusive_by="fov_id",
  n_proteins=50,
)

#### 6.3.4.2. Accessing the Dataset Internals


## 6.4. Building a Prototype Model
### 6.4.1. Defining the LocalizationModel


In [ ]:
from dlfb_pytorch.localization.model import LocalizationModel

display([LocalizationModel])

### 6.4.2. The Encoder: Processing Input Images


In [ ]:
from dlfb_pytorch.localization.model import Encoder, ResnetBlock

display([Encoder, ResnetBlock])

### 6.4.3. The VectorQuantizer: Discretizing the Embeddings


In [ ]:
from dlfb_pytorch.localization.model import VectorQuantizer

display([VectorQuantizer])

In [ ]:
display([VectorQuantizer.__call__])

In [ ]:
display([VectorQuantizer.quantize, VectorQuantizer.calculate_distances])

#### 6.4.3.1. Calculating VQ-VAE–Specific Losses


In [ ]:
display([VectorQuantizer.compute_losses])

#### 6.4.3.2. Using Perplexity to Measure Codebook Use


In [ ]:
display([VectorQuantizer.calculate_perplexity])

#### 6.4.3.3. Using the Straight-Through Estimator


In [ ]:
display([VectorQuantizer.get_straight_through_estimator])

### 6.4.4. Decoder: Decoding the Discretized Embeddings Back to Images


In [ ]:
from dlfb_pytorch.localization.model import Decoder, Upsample

display([Decoder, Upsample])

### 6.4.5. ClassificationHead: A Simple but Crucial Module


In [ ]:
from dlfb_pytorch.localization.model import ClassificationHead

display([ClassificationHead])

### 6.4.6. Setting Up Model Training


In [ ]:
from dlfb_pytorch.localization.train import train

display([train])

In [ ]:
from dlfb_pytorch.localization.dataset import Dataset

display([Dataset.get_batches])

In [ ]:
from dlfb_pytorch.localization.train import train_step

display([train_step])

## 6.5. Training with a Small Image Set


In [ ]:
np.random.seed(42)

dataset_splits = DatasetBuilder(
  data_path=assets("localization/datasets")
).build(
  splits={"train": 0.80, "valid": 0.10, "test": 0.10},
  n_proteins=50,
)

In [ ]:
from dlfb_pytorch.localization.dataset.utils import count_unique_proteins
from dlfb_pytorch.localization.model import LocalizationModel
from dlfb_pytorch.localization.train import train

model = LocalizationModel(
  num_classes=count_unique_proteins(dataset_splits),
  embedding_dim=64,
  num_embeddings=512,
  commitment_cost=0.25,
  dropout_rate=0.45,
  classification_head_layers=2,
).to(device)

In [ ]:
optimizer = model.create_optimizer(lr=0.001)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_epochs=10,
  batch_size=256,
  classification_weight=1,
  eval_every=1,
  store_path=assets("localization/models/small"),
)

### 6.5.1. Inspecting Image Reconstruction


In [ ]:
from dlfb_pytorch.localization.inspect.reconstruction import show_reconstruction

np.random.seed(42)
show_reconstruction(dataset_splits["valid"], model, n=8);

### 6.5.2. Evaluation Metrics over Epochs


In [ ]:
from dlfb_pytorch.localization.inspect.metrics import plot_losses

plot_losses(metrics);

In [ ]:
from dlfb_pytorch.localization.inspect.metrics import plot_perplexity

plot_perplexity(metrics);

### 6.5.3. Model without Classification Task


In [ ]:
model_alt = LocalizationModel(
  num_classes=count_unique_proteins(dataset_splits),
  embedding_dim=64,
  num_embeddings=512,
  commitment_cost=0.25,
  dropout_rate=0.45,
  classification_head_layers=2,
).to(device)

optimizer_alt = model_alt.create_optimizer(lr=0.001)

model_alt, optimizer_alt, metrics_alt = train(
  model=model_alt,
  optimizer=optimizer_alt,
  dataset_splits=dataset_splits,
  num_epochs=10,
  batch_size=256,
  classification_weight=0,  # i.e. the protein id are ignored
  eval_every=1,
  store_path=assets("localization/models/small_alt"),
)

In [ ]:
plot_perplexity(metrics_alt);

In [ ]:
show_reconstruction(dataset_splits["valid"], state_alt, n=8, rng=rng_frames);

## 6.6. Understanding the Model
### 6.6.1. Understanding Localization Clustering


In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.utils import get_frame_embeddings

display([get_frame_embeddings])

In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.clustering import (
  calculate_projection,
  plot_projection,
)
from dlfb_pytorch.localization.inspect.embeddings.utils import get_frame_embeddings

frame_embeddings = {}
for name, m in zip(["no_head", "with_head"], [model_alt, model]):
  frame_embeddings[name] = get_frame_embeddings(m, dataset_splits["valid"])

projection = calculate_projection(frame_embeddings)
plot_projection(
  projection,
  dataset_splits["valid"],
  titles=["No ClassificationHead", "With ClassificationHead"],
);

### 6.6.2. Inspecting Feature Spectrums


In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.utils import cluster_feature_spectrums

display([cluster_feature_spectrums])

In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.feature_spectrum import (
  plot_encoding_corr_heatmap,
)
from dlfb_pytorch.localization.inspect.embeddings.utils import aggregate_proteins

protein_ids, protein_histograms = aggregate_proteins(
  dataset_splits["valid"], **frame_embeddings["with_head"]
)
corr_idx_idx, tree, encoding_clusters = cluster_feature_spectrums(
  protein_histograms, n_clusters=8
)
plot_encoding_corr_heatmap(corr_idx_idx, tree, encoding_clusters);

In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.feature_spectrum import (
  plot_stacked_histrograms,
)
from dlfb_pytorch.localization.inspect.embeddings.utils import aggregate_localizations

localizations, localization_histograms = aggregate_localizations(
  dataset_splits["valid"], protein_ids, protein_histograms
)
plot_stacked_histrograms(
  localizations, localization_histograms, tree, encoding_clusters
);

## 6.7. Improving the Model
### 6.7.1. Scaling Up the Data


In [ ]:
np.random.seed(42)

dataset_splits = DatasetBuilder(
  data_path=assets("localization/datasets")
).build(
  splits={"train": 0.80, "valid": 0.10, "test": 0.10},
  n_proteins=500,  # a larger number of proteins
)

model = LocalizationModel(
  num_classes=count_unique_proteins(dataset_splits),
  embedding_dim=64,
  num_embeddings=512,
  commitment_cost=0.25,
  dropout_rate=0.45,
  classification_head_layers=2,
).to(device)

optimizer = model.create_optimizer(lr=0.001)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_epochs=10,
  batch_size=256,
  classification_weight=1,
  eval_every=1,
  store_path=assets("localization/models/large"),
)

In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.feature_spectrum import (
  plot_stacked_histrograms,
)
from dlfb_pytorch.localization.inspect.embeddings.utils import (
  aggregate_localizations,
  aggregate_proteins,
  cluster_feature_spectrums,
  get_frame_embeddings,
)

frame_embeddings = get_frame_embeddings(model, dataset_splits["valid"])
protein_ids, protein_histograms = aggregate_proteins(
  dataset_splits["valid"], **frame_embeddings
)
_, tree, encoding_clusters = cluster_feature_spectrums(
  protein_histograms, n_clusters=12
)
localizations, localization_histograms = aggregate_localizations(
  dataset_splits["valid"], protein_ids, protein_histograms
)
plot_stacked_histrograms(
  localizations, localization_histograms, tree, encoding_clusters
);

In [ ]:
from dlfb_pytorch.localization.inspect.embeddings.clustering import (
  calculate_projection,
  plot_projection,
)

projection = calculate_projection(frame_embeddings)
plot_projection(
  projection,
  dataset_splits["valid"],
  subset_mode="single",  # Only show frames with single localization
  titles=["Localization UMAP Projection"],
);

### 6.7.2. Going Further


## 6.8. Summary
